# Day 7

In [1]:
import fs from 'node:fs';

In [2]:
const input = fs.readFileSync("input.txt", "utf-8");

In [3]:
const sample = `\
.......S.......
...............
.......^.......
...............
......^.^......
...............
.....^.^.^.....
...............
....^.^...^....
...............
...^.^...^.^...
...............
..^...^.....^..
...............
.^.^.^.^.^...^.
...............`

## Part 1

Okay, in the first part of the day, the problem is as follows: The input is a grid. The cell with `S` emits a beam downwards. Everytime a beam hits a splitter `^` the beam "splits" to the right and left of the splitter, ie
```
.|.
|^|
|.|
```
We need to count how many splitters were hit. Now, I have some ideas for considering the splitters as logic circuits, but the simplest solution would be to simulate the beam and then count the splitters that were hit.

In [4]:
const rows = sample.split("\n")

In [7]:
[rows[0], rows[1]]

[ ".......S.......", "..............." ]

I think given the previous row after simulating and the current row, we can calculate the current simulated row using these rules:
- `.`
    -  if above it there's a `|` or `S` then becomes `|`
    -  if to the right/left of it there's a `^` with `|` above then it becomes `|`
- `^`: stays the same

In [22]:
const w = rows[0].length;
const prev = rows[0];
const cur = rows[1];
const sim = []
for (let i = 0; i < w; i++) {
  if (cur[i] === "^") sim.push("^");
  else if (cur[i] === ".") {
    if (prev[i] === "S" || prev[i] === "|") {
      sim.push("|");
    }
    else if (0 < i && cur[i-1] === "^" && prev[i-1] === "|") {
      sim.push("|");
    }
    else if (i < w - 1 && cur[i+1] === "^" && prev[i+1] === "|") {
      sim.push("|");
    }
    else {
      sim.push(".")
    }
  }
  else {
    console.log(cur[i], "something went wrong")
  }
}

console.log(prev);
console.log(sim.join(""));

.......S.......
.......|.......


In [24]:
function simulate_step(prev: string, cur: string) {
  const w = prev.length;
  if (cur.length != w) throw new Exception("prev and cur lengths don't match");
  const sim = []
  for (let i = 0; i < w; i++) {
    if (cur[i] === "^") sim.push("^");
    else if (cur[i] === ".") {
      if (prev[i] === "S" || prev[i] === "|") {
        sim.push("|");
      }
      else if (0 < i && cur[i-1] === "^" && prev[i-1] === "|") {
        sim.push("|");
      }
      else if (i < w - 1 && cur[i+1] === "^" && prev[i+1] === "|") {
        sim.push("|");
      }
      else {
        sim.push(".")
      }
    }
    else {
      console.log(cur[i], "something went wrong")
    }
  }
  return sim.join("");
}

simulate_step(rows[0], rows[1])

".......|......."

In [31]:
const sim_rows = [rows[0]];
for (let i = 1; i < rows.length; i++) {
  const sim = simulate_step(sim_rows[sim_rows.length-1], rows[i]);
  sim_rows.push(sim);
}
sim_rows.forEach(r => console.log(r))

.......S.......
.......|.......
......|^|......
......|.|......
.....|^|^|.....
.....|.|.|.....
....|^|^|^|....
....|.|.|.|....
...|^|^|||^|...
...|.|.|||.|...
..|^|^|||^|^|..
..|.|.|||.|.|..
.|^|||^||.||^|.
.|.|||.||.||.|.
|^|^|^|^|^|||^|
|.|.|.|.|.|||.|


In [33]:
let sum = 0;
for (let i = 1; i < sim_rows.length; i++) {
  for (let j = 0; j < sim_rows[i].length; j++) {
    if (sim_rows[i][j] === "^" && sim_rows[i-1][j] === "|") sum += 1;
  }
}
sum;

21

In [34]:
function part1(input) {
  const rows = input.split("\n");
  const sim_rows = [rows[0]];
  for (let i = 1; i < rows.length; i++) {
    const sim = simulate_step(sim_rows[sim_rows.length-1], rows[i]);
    sim_rows.push(sim);
  }
  
  let sum = 0;
  for (let i = 1; i < sim_rows.length; i++) {
    for (let j = 0; j < sim_rows[i].length; j++) {
      if (sim_rows[i][j] === "^" && sim_rows[i-1][j] === "|") sum += 1;
    }
  }
  return sum;
}

part1(sample);

21

In [35]:
part1(input);

1541

## Part 2

Now things get interesting. Instead of a beam we have a "quantum particle". Instead of splitting the particle can either go left or right. Each choice creates a different trajectory, and the question then becomes, how many different trajectories are there. Now, in general, each choice is binary so it grows exponentially, but in reality it also depends on where the particle was, ie
```
|..
..^
...
```
This scenario has one trajectory
```
|..
^..
...
```
This one also has one
```
.|.
.^.
...
```
This has two trajectories
```
..|..
..^..
.....
.^...
.....
```
and this one has three.

When a particle hits a splitter, we don't really need to know the trajectory it took, the only thing that matters are that the number of trajectories afterwards will be doubled, half going right and half going left. Therefore, we can continue with the simulation iterative approach, but instead of beams will replace each cell with a number for the number of trajectories where the particle is in that cell. To illustrate, the last example will be converted to
```
00100
01010
01010
10110
10110
```
Then the total number of trajectories will be the sum in the last cell. Okay, this does mean that we can use strings anymore since I expect the number of trajectories to be larger than 9.

In [36]:
rows[0]

".......S......."

In [40]:
const initial = rows[0].split('').map(c => c === "S" ? 1 : 0);
initial

[
  0, 0, 0, 0, 0, 0,
  0, 1, 0, 0, 0, 0,
  0, 0, 0
]

In [56]:
function simulate_step2(prev: Array<num>, cur: string) {
  const w = prev.length;
  if (cur.length != w) throw new Exception("prev and cur lengths don't match");
  const count = prev.map(n => 0);
  for (let i = 0; i < w; i++) {
    if (cur[i] === "^") count[i] = 0;
    else if (cur[i] === ".") {
      count[i] += prev[i];
      if (0 < i && cur[i-1] === "^") {
        count[i] += prev[i-1];
      }
      
      if (i < w - 1 && cur[i+1] === "^") {
        count[i] += prev[i+1];
      }
    }
    else {
      console.log(cur[i], "something went wrong")
    }
  }
  return count;
}

simulate_step2(initial, rows[2]).join("")

"000000101000000"

In [53]:
function part2(input) {
  const rows = input.split("\n");
  let count = rows[0].split('').map(c => c === "S" ? 1 : 0);
  for (let i = 1; i < rows.length; i++) {
    count = simulate_step2(count, rows[i]);
  }
  
  return count.reduce((acc, x) => acc + x);
}

part2(sample);

40

In [54]:
part2(input);

80158285728929